# LangChain (open source): Chains

### Outline
- `LLMChain` equivalent
- Sequential chains: `SimpleSequentialChain` and `SequentialChain` equivalents
- Router chain equivalent

Open-source recode of `02-LangChain-for-LLM-Application-Development/L3-chains.ipynb`.
`LLMChain`, `SimpleSequentialChain`, `SequentialChain`, `MultiPromptChain` and
`LLMRouterChain` were all removed - `langchain.chains` doesn't exist anymore in this LangChain
version. LCEL (the `|` operator) is now the only way to build chains:

- `LLMChain` -> `prompt | model | StrOutputParser()`.
- `SimpleSequentialChain` (single in, single out) -> pipe one chain's string output straight
  into the next chain's input dict.
- `SequentialChain` (multiple named inputs/outputs) -> a stack of
  `RunnablePassthrough.assign(key=chain)` calls, each adding one more key to a running dict.
- The router chain -> `model.with_structured_output(RouterDecision)` picks a destination name
  (LLM-as-classifier instead of a hand-parsed markdown JSON blob), then a plain Python dict
  lookup dispatches to the chosen chain.

`Data.csv` from the original isn't included in this repo - `data/reviews.csv` is a small
synthetic stand-in with the same shape (Product, Review columns, one review in French to
exercise the translate/detect-language chains).

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`open_source_agentic_ai_course`](../../open_source_agentic_ai_course/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [ ]:
import sys
from pathlib import Path

# common.py / tracing.py live in open_source_agentic_ai_course/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../open_source_agentic_ai_course").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

import pandas as pd
from common import get_model, traced
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

In [ ]:
df = pd.read_csv(COURSE_DIR / "data" / "reviews.csv")
df.head()

In [ ]:
model = get_model(temperature=0.9)

## `LLMChain` equivalent

`prompt | model | StrOutputParser()` - a prompt template piped into a model, piped into a
string-output parser.

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain = prompt | model | StrOutputParser()

product = "Queen Size Sheet Set"
print(chain.invoke({"product": product}, config=traced("L3: LLMChain equivalent")))

## `SimpleSequentialChain` equivalent

Single input, single output, piped straight from one chain into the next.

In [ ]:
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain_one = first_prompt | model | StrOutputParser()

second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 word description for the following company: {company_name}"
)
chain_two = second_prompt | model | StrOutputParser()

overall_simple_chain = chain_one | (lambda company_name: {"company_name": company_name}) | chain_two

print(overall_simple_chain.invoke({"product": product}, config=traced("L3: SimpleSequentialChain equivalent")))

## `SequentialChain` equivalent

Multiple named inputs/outputs: translate a review, summarize it, detect its language, then write
a follow-up in that language - each step reads keys earlier steps added and adds its own.
`RunnablePassthrough.assign(key=chain)` is the LCEL building block: it runs `chain` on the
current dict and merges its output back in under `key`, so the dict keeps growing.

In [ ]:
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{review}"
)
chain_translate = translate_prompt | model | StrOutputParser()

summarize_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
chain_summarize = summarize_prompt | model | StrOutputParser()

language_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{review}"
)
chain_language = language_prompt | model | StrOutputParser()

followup_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_followup = followup_prompt | model | StrOutputParser()

### How `overall_chain` is built

`RunnablePassthrough.assign(key=chain)` takes whatever dict is flowing through the pipe, runs
`chain` on it, and merges the result back into the dict under `key` - it never removes the keys
already there, it only adds one. Chaining several `.assign(...)` with `|` means each step sees
every key every earlier step added:

1. `.assign(English_Review=chain_translate)` - input is `{"review": ...}`; `chain_translate`
   reads `{review}` and its output is merged in as `English_Review`.
2. `.assign(summary=chain_summarize)` - `chain_summarize` reads `{English_Review}` (added in
   step 1) and adds `summary`.
3. `.assign(language=chain_language)` - `chain_language` reads `{review}` (the original input,
   still present) and adds `language`.
4. `.assign(followup_message=chain_followup)` - `chain_followup` reads both `{summary}` and
   `{language}` (from steps 2 and 3) and adds the final `followup_message`.

By the end, the dict carries all five keys - `review`, `English_Review`, `summary`, `language`,
`followup_message`. That growing-dict shape is exactly what the removed
`SequentialChain(input_variables=..., output_variables=...)` did, just without having to
declare the variable names up front: each `.assign` simply adds one more key to whatever is
already there.

In [ ]:
overall_chain = (
    RunnablePassthrough.assign(English_Review=chain_translate)
    | RunnablePassthrough.assign(summary=chain_summarize)
    | RunnablePassthrough.assign(language=chain_language)
    | RunnablePassthrough.assign(followup_message=chain_followup)
)

In [ ]:
review = df.Review[5]
result = overall_chain.invoke({"review": review}, config=traced("L3: SequentialChain equivalent"))
for key in ("English_Review", "summary", "language", "followup_message"):
    print(f"\n{key}:\n{result[key]}")

## Router chain equivalent

Four specialist prompts (physics, math, history, computer science) plus a default. The original
notebook has the LLM emit a markdown-wrapped JSON blob naming the destination, hand-parsed by
`RouterOutputParser`. The modern version just asks for a `RouterDecision` via
`with_structured_output` - the model directly returns a validated `destination` +
`next_input`, no string parsing involved.

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise \
and easy to understand manner. When you don't know the answer to a \
question you admit that you don't know.

Here is a question:
{input}"""

math_template = """You are a very good mathematician. \
You are great at answering math questions. You are so good because \
you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the \
broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people, events \
and contexts from a range of historical periods. You have the ability \
to think, reflect, debate, discuss and evaluate the past. You have a \
respect for historical evidence and the ability to make use of it to \
support your explanations and judgements.

Here is a question:
{input}"""

computerscience_template = """You are a successful computer scientist. \
You have a passion for creativity, collaboration, forward-thinking, \
confidence, strong problem-solving capabilities, understanding of \
theories and algorithms, and excellent communication skills. You are \
great at answering coding questions.

Here is a question:
{input}"""

prompt_infos = [
    {"name": "physics", "description": "Good for answering questions about physics", "template": physics_template},
    {"name": "math", "description": "Good for answering math questions", "template": math_template},
    {"name": "History", "description": "Good for answering history questions", "template": history_template},
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "template": computerscience_template,
    },
]

In [ ]:
destination_chains = {
    info["name"]: ChatPromptTemplate.from_template(info["template"]) | model | StrOutputParser()
    for info in prompt_infos
}
default_chain = ChatPromptTemplate.from_template("{input}") | model | StrOutputParser()

destinations_str = "\n".join(f"{info['name']}: {info['description']}" for info in prompt_infos)


class RouterDecision(BaseModel):
    """Pick which specialist prompt should answer the question."""

    destination: str = Field(
        description=f"One of the candidate prompt names, or DEFAULT if none fit well:\n{destinations_str}"
    )
    next_input: str = Field(description="The question to send to that destination, possibly reworded for clarity")


router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given a raw text input to a language model, select the model prompt best suited "
            "for the input. You may also revise the original input if you think that revising it "
            "will ultimately lead to a better response from the language model.\n\n"
            f"Candidate prompts:\n{destinations_str}",
        ),
        ("user", "{input}"),
    ]
)
router_chain = router_prompt | model.with_structured_output(RouterDecision, method="function_calling")


def route(decision: RouterDecision, config=None) -> str:
    chain = destination_chains.get(decision.destination, default_chain)
    return chain.invoke({"input": decision.next_input}, config=config)

In [ ]:
for question in ["What is black body radiation?", "what is 2 + 2", "Why does every cell in our body contain DNA?"]:
    decision = router_chain.invoke({"input": question}, config=traced(f"L3: router decision - {question[:30]}"))
    print(f"\n> {question}")
    print(f"routed to: {decision.destination!r}")
    print(route(decision, config=traced(f"L3: router answer - {question[:30]}")))